# 🤖 Gemma 4 도구사용(Tool-Calling) 에이전트 — Colab 학습판

HuggingFace Inference Providers(**Novita**)를 통해 **Gemma 4**를 호출하고,
모델이 필요할 때 **로컬 파일을 직접 읽는** 미니 에이전트를 처음부터 만든다.

## 🧠 핵심 원리 (한 장 요약)
> **모델(뇌)은 컴퓨터를 못 만진다. 실제 일은 우리 코드(손)가 한다.**
```
① 나: "이 폴더 뭐 있어?"   (질문 + 도구목록 함께 전송)
② Gemma: 답 대신 → "list_directory() 실행해줘"  (그냥 JSON, 실행 아님)
③ 코드: 진짜로 os.listdir() 실행 → 결과를 'tool' 메시지로 되먹임
④ Gemma: 결과를 읽고 → "A, B, C 파일이 있습니다"  최종 답변
```
위에서부터 셀을 하나씩 실행하세요.

In [ ]:
# 1) 설치
!pip install openai -q

## 2) 토큰 설정
HuggingFace 토큰(**Fine-grained** + *Make calls to Inference Providers* 권한)이 필요합니다.
- **권장**: Colab 왼쪽 🔑(Secrets)에 이름 `HF_TOKEN` 으로 등록 → 아래 셀이 자동으로 읽음
- 없으면 실행 시 입력창이 뜹니다(getpass, 화면에 안 보이게).

In [ ]:
# 2) 토큰 로드
import os
try:
    from google.colab import userdata           # Colab Secrets 우선
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN: Colab Secrets 에서 로드됨")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN 붙여넣기: ")
    print("HF_TOKEN: 입력값 사용")

## 3) [STEP 1] 가장 단순한 호출 — 딱 3단계
① 클라이언트(주소+토큰) → ② 메시지 전송 → ③ 답 출력. 이것만 이해하면 절반은 끝.

In [ ]:
# 3) STEP 1 — 단발 호출
from openai import OpenAI

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])
resp = client.chat.completions.create(
    model="google/gemma-4-31B-it:novita",        # :novita = Novita 제공자 지정
    messages=[{"role": "user", "content": "자기소개를 한 문장으로 해줘."}],
)
print("🤖", resp.choices[0].message.content)

## 4) [STEP 2] 도구(tools) 정의 — '손'
모델에게 쥐여줄 로컬 함수들. 파일 도구는 **ALLOWED_ROOT 안으로만 제한**되고 `.env` 같은 비밀파일은 차단된다.
`@tool` 하나만 붙이면 자동 등록 — 이게 확장 지점이다.

In [ ]:
# 4) 도구 정의
import os

# 안전장치
_MAX_BYTES = 100_000
ALLOWED_ROOT = os.path.realpath(os.getcwd())      # Colab은 보통 /content (필요시 넓히세요)
_DENY_NAMES = {".env"}
_DENY_SUFFIXES = (".pem", ".key")
_DENY_CONTAINS = ("id_rsa",)

_REGISTRY = []

def tool(name, description, parameters=None, required=None):
    if any(s["function"]["name"] == name for s, _ in _REGISTRY):
        raise ValueError(f"중복된 도구 이름: {name!r}")          # 조용한 덮어쓰기 방지
    params = {"type": "object", "properties": parameters or {}}
    if required:
        params["required"] = required
    schema = {"type": "function",
              "function": {"name": name, "description": description, "parameters": params}}
    def deco(fn):
        _REGISTRY.append((schema, fn)); return fn
    return deco

def _safe_path(path):
    full = os.path.realpath(path)
    root = os.path.realpath(ALLOWED_ROOT)
    if full != root and not full.startswith(root + os.sep):
        return None                                # 루트 밖 차단
    base = os.path.basename(full).lower()
    if base in _DENY_NAMES or base.endswith(_DENY_SUFFIXES) or any(x in base for x in _DENY_CONTAINS):
        return None                                # 비밀파일 차단
    return full

@tool("get_current_directory", "현재 작업 디렉토리(절대경로)를 반환한다.")
def get_current_directory():
    return os.getcwd()

@tool("list_directory", "지정 경로의 파일/폴더 목록을 반환한다. path 생략 시 현재 폴더.",
      parameters={"path": {"type": "string", "description": "조회할 경로 (기본 '.')"}})
def list_directory(path="."):
    safe = _safe_path(path)
    if safe is None:
        return "오류: 접근이 허용되지 않은 경로입니다."
    try:
        rows = []
        for n in sorted(os.listdir(safe)):
            full = os.path.join(safe, n)
            rows.append(f"[폴더] {n}" if os.path.isdir(full)
                        else f"[파일] {n} ({os.path.getsize(full):,} bytes)")
        return "\n".join(rows) or "(빈 폴더)"
    except Exception as e:
        return f"오류: {e}"

@tool("read_file", "지정한 텍스트 파일의 내용을 반환한다.",
      parameters={"path": {"type": "string", "description": "읽을 파일 경로"}},
      required=["path"])
def read_file(path):
    safe = _safe_path(path)
    if safe is None:
        return "오류: 접근이 허용되지 않은 경로입니다."
    try:
        if os.path.getsize(safe) > _MAX_BYTES:
            return f"오류: 파일이 너무 큽니다(>{_MAX_BYTES:,} bytes)."
        with open(safe, encoding="utf-8", errors="replace") as f:
            return f.read()
    except Exception as e:
        return f"오류: {e}"

TOOLS = [schema for schema, _ in _REGISTRY]
TOOL_FUNCS = {schema["function"]["name"]: fn for schema, fn in _REGISTRY}
print(f"도구 {len(TOOLS)}개 로드:", list(TOOL_FUNCS))

## 5) [STEP 3] 에이전트 루프 — '뇌↔손' 연결
도구 요청이 없어질 때까지(최대 `MAX_STEPS`) 반복하며, 모델이 요청한 도구를 실행해 결과를 되먹인다.
**안전장치 내장**: 무한루프 상한, 도구 예외 흡수(세션 안 깨짐), 거대출력 절단, 빈 도구목록 미전송.

In [ ]:
# 5) 에이전트 루프
import json
from openai import OpenAI

MODEL = "google/gemma-4-31B-it:novita"
SYSTEM_PROMPT = ("너는 로컬 환경을 다룰 수 있는 한국어 AI 어시스턴트야. "
                 "파일·데이터 질문은 추측하지 말고 제공된 도구를 호출해 확인한 뒤 답해. 간결하게.")
MAX_STEPS = 10
MAX_TOOL_OUTPUT_CHARS = 20_000

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])

def _run_tool(name, args, tool_funcs):
    fn = tool_funcs.get(name)
    if fn is None:
        return f"오류: 알 수 없는 도구 '{name}'"
    try:
        return str(fn(**args))
    except Exception as e:
        return f"오류: 도구 '{name}' 실행 실패: {e}"      # 예외를 문자열로 흡수

def agent_reply(messages, client, *, model=MODEL, tools=None, tool_funcs=None, max_steps=MAX_STEPS):
    if tools is None: tools = TOOLS
    if tool_funcs is None: tool_funcs = TOOL_FUNCS
    for _ in range(max_steps):
        kwargs = {"model": model, "messages": messages}
        if tools: kwargs["tools"] = tools                 # 빈 배열이면 미전송(400 방지)
        msg = client.chat.completions.create(**kwargs).choices[0].message
        assistant = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments or "{}"}}
                for tc in msg.tool_calls]
        messages.append(assistant)
        if not msg.tool_calls:
            return msg.content or ""                       # None 방지
        for tc in msg.tool_calls:                          # 모든 tool_call 에 반드시 응답
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            result = _run_tool(tc.function.name, args, tool_funcs)
            if len(result) > MAX_TOOL_OUTPUT_CHARS:
                result = result[:MAX_TOOL_OUTPUT_CHARS] + f"\n...[잘림: 전체 {len(result):,}자]"
            print(f"   🔧 {tc.function.name}({args}) 실행")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "⚠️ 도구 호출이 너무 많아 중단했습니다."
print("agent_reply 준비 완료")

## 6) 오프라인 안전성 점검 (토큰 불필요)
방어장치가 실제로 동작하는지 확인한다 — API 호출 없이.

In [ ]:
# 6) 안전성 점검
import types

# (1) 프로젝트 밖 경로 차단
assert read_file("/etc/passwd").startswith("오류:")

# (2) 중복 도구 이름 거부
try:
    tool("read_file", "dup")(lambda: 1)
    raise AssertionError("중복이 통과됨")
except ValueError:
    pass

# (3) 가짜 client 로 '도구 예외 → 세션 안 깨짐' 검증
def _tc(i, n, a):
    return types.SimpleNamespace(id=i, type="function",
                                 function=types.SimpleNamespace(name=n, arguments=a))
def _resp(c=None, tcs=None):
    return types.SimpleNamespace(
        choices=[types.SimpleNamespace(message=types.SimpleNamespace(content=c, tool_calls=tcs))])
class Fake:
    def __init__(self, r): self.r = list(r); self.calls = []
    @property
    def chat(self):
        outer = self
        class C:
            def create(self, **k):
                outer.calls.append(k)
                return outer.r.pop(0) if outer.r else _resp("done")
        return types.SimpleNamespace(completions=C())

def boom(**k):
    raise RuntimeError("고장")
msgs = [{"role": "user", "content": "hi"}]
out = agent_reply(msgs, Fake([_resp(tcs=[_tc("1", "boom", "{}")]), _resp("복구됨")]),
                  tools=[{"type": "function", "function": {"name": "boom",
                          "parameters": {"type": "object", "properties": {}}}}],
                  tool_funcs={"boom": boom})
assert any(m.get("role") == "tool" and m["tool_call_id"] == "1" for m in msgs)  # 고아 없음
assert out == "복구됨"
print("✅ 안전성 점검 통과: 경로제한 · 중복거부 · 예외복구")

## 7) 대화 실행 🎉
아래 셀을 실행하면 대화가 시작된다. `exit`/`종료` 입력 시 끝.
예: **"지금 폴더에 뭐 있어?"**, **"README.md 읽어줘"** (Colab에선 /content 기준).

In [ ]:
# 7) 대화 (exit / quit / 종료 로 종료)
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
print(f"💬 Gemma 4 에이전트 시작 | 도구 {len(TOOLS)}개 (종료: exit/quit/종료)")
while True:
    try:
        user_input = input("\n🧑 나: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 종료"); break
    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "종료"):
        print("👋 종료"); break
    messages.append({"role": "user", "content": user_input})
    try:
        print(f"🤖 Gemma: {agent_reply(messages, client)}")
    except Exception as e:
        print(f"❌ 호출 실패: {e}  (401=토큰 / 402=크레딧 / 403=권한)")

## 8) 다음 단계 (확장)
- **새 도구 추가**: `@tool(...)` 함수 하나 붙이면 자동 인식. 파일 도구는 반드시 `_safe_path()` 로 경로 제한.
- **아이디어**: `get_current_datetime`(오늘 날짜), `read_table`(CSV/Excel), `run_sql`(DB SELECT), `web_search`.
- **비용**: 무료 $0.10/월. 짧은 요청 ~190회. 초과해도 402로 멈출 뿐 자동청구 없음.
- 로컬 버전(파일 분리 + pytest 테스트)은 GitHub 비공개 저장소에 있음.